# 06 - Modélisation ML : Prédiction blocs électoraux 2025–2027

**Objectif** : Prédire le `bloc_dominant` (Gauche / Centre / Droite / ExtremeDroite) par commune pour 2025, 2026 et 2027.

**Données** : `gold.features_communes` — 620 lignes, 124 communes × 5 années (2002, 2007, 2012, 2017, 2022)

**Approche** :
- Split temporel : train ≤ 2017, test = 2022
- Modèles : Random Forest + XGBoost
- Features : CSP, diplômes, démographie, tendances temporelles (lag, évolution)
- Prédictions : extrapolation tendancielle 2025, 2026, 2027

## 0. Configuration & imports

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Configuration inline
PG_HOST     = os.environ.get('POSTGRES_HOST',     'postgres')
PG_PORT     = os.environ.get('POSTGRES_PORT',     '5432')
PG_DB       = os.environ.get('POSTGRES_DB',       'mspr813')
PG_USER     = os.environ.get('POSTGRES_USER',     'mspr_user')
PG_PASSWORD = os.environ.get('POSTGRES_PASSWORD', 'mspr_password')

DB_URL = f'postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}'
engine = create_engine(DB_URL, pool_pre_ping=True)

print('PostgreSQL OK :', engine.connect().execute(text('SELECT current_database()')).scalar())

PostgreSQL OK : mspr813


## 1. Chargement des données Gold

In [2]:
df = pd.read_sql('SELECT * FROM gold.features_communes ORDER BY code_commune, annee', engine)

print(f'Shape : {df.shape}')
print(f'Communes : {df["code_commune"].nunique()}')
print(f'Années   : {sorted(df["annee"].unique())}')
print(f'Blocs    : {df["bloc_dominant"].value_counts().to_dict()}')
df.head(3)

Shape : (620, 23)
Communes : 124
Années   : [2002, 2007, 2012, 2017, 2022]
Blocs    : {'Gauche': 397, 'Droite': 157, 'Centre': 64, 'ExtremeDroite': 2}


,code_commune,annee,libelle,code_dep,pct_gauche,pct_centre,pct_droite,pct_extremedroite,pct_divers,bloc_dominant,...,mediane_revenu_disp,gini,taux_pauvrete,cadres_pct,ouvriers_pct,employes_pct,artisans_pct,pct_bac_plus,pct_sans_diplome,created_at
0,75056,2002,PARIS,75,37.057,14.534,35.579,12.830,0.0,Gauche,...,None,None,None,NaN,NaN,NaN,NaN,65.439,11.511,2026-03-04 09:50:24.128398
1,75056,2007,PARIS,75,37.615,20.732,37.078,4.576,0.0,Gauche,...,None,None,None,NaN,NaN,NaN,NaN,65.439,11.511,2026-03-04 09:50:24.128398
2,75056,2012,PARIS,75,51.044,9.337,33.423,6.197,0.0,Gauche,...,None,None,None,NaN,NaN,NaN,NaN,65.439,11.511,2026-03-04 09:50:24.128398


## 2. Feature Engineering temporel

Ajout de features dérivées : tendances entre élections, lag.

In [3]:
df = df.sort_values(['code_commune', 'annee']).reset_index(drop=True)

# Lag : résultat de l'élection précédente (par commune)
for col in ['pct_gauche', 'pct_centre', 'pct_droite', 'pct_extremedroite']:
    df[f'{col}_lag1'] = df.groupby('code_commune')[col].shift(1)

# Tendance : évolution par rapport à t-1
for col in ['pct_gauche', 'pct_centre', 'pct_droite', 'pct_extremedroite']:
    df[f'{col}_trend'] = df[col] - df[f'{col}_lag1']

# Bloc dominant à t-1 (encodé)
df['bloc_lag1'] = df.groupby('code_commune')['bloc_dominant'].shift(1)

# Intervalle depuis la dernière élection (années)
df['annee_lag1'] = df.groupby('code_commune')['annee'].shift(1)
df['intervalle'] = df['annee'] - df['annee_lag1']

print(f'Shape après features temporelles : {df.shape}')
df[['code_commune','annee','pct_gauche','pct_gauche_lag1','pct_gauche_trend','bloc_lag1']].head(10)

Shape après features temporelles : (620, 34)


,code_commune,annee,pct_gauche,pct_gauche_lag1,pct_gauche_trend,bloc_lag1
0,75056,2002,37.057,NaN,NaN,NaN
1,75056,2007,37.615,37.057,0.558,Gauche
2,75056,2012,51.044,37.615,13.429,Gauche
3,75056,2017,30.634,51.044,-20.410,Gauche
4,75056,2022,42.313,30.634,11.679,Centre
5,92002,2002,35.115,NaN,NaN,NaN
6,92002,2007,34.422,35.115,-0.693,Gauche
7,92002,2012,45.819,34.422,11.397,Droite
8,92002,2017,26.322,45.819,-19.497,Gauche
9,92002,2022,35.509,26.322,9.187,Centre


In [4]:
# Définir les features et la target
FEATURES = [
    # Résultats précédents (lag)
    'pct_gauche_lag1', 'pct_centre_lag1', 'pct_droite_lag1', 'pct_extremedroite_lag1',
    # Tendances
    'pct_gauche_trend', 'pct_centre_trend', 'pct_droite_trend', 'pct_extremedroite_trend',
    # CSP
    'cadres_pct', 'ouvriers_pct', 'employes_pct', 'artisans_pct',
    # Diplômes
    'pct_bac_plus', 'pct_sans_diplome',
    # Démographie
    'nb_naissances', 'nb_deces', 'solde_naturel',
    # Temporel
    'annee',
]

TARGET = 'bloc_dominant'

# Supprimer les lignes sans lag (première élection par commune)
df_ml = df[df['bloc_lag1'].notna()].copy()
print(f'Lignes avec lag disponible : {len(df_ml)} (sur {len(df)} total)')
print(f'Années dans df_ml : {sorted(df_ml["annee"].unique())}')

Lignes avec lag disponible : 496 (sur 620 total)
Années dans df_ml : [2007, 2012, 2017, 2022]


## 3. Split temporel : train ≤ 2017 / test = 2022

In [5]:
# Split : train sur 2002/2007/2012, test sur 2017
# La prediction cible sera 2022 (comparaison avec les resultats reels 2022)
df_train = df_ml[df_ml['annee'] <= 2012].copy()
df_test  = df_ml[df_ml['annee'] == 2017].copy()

print(f'Train : {len(df_train)} lignes | annees : {sorted(df_train["annee"].unique())}')
print(f'Test  : {len(df_test)}  lignes | annees : {sorted(df_test["annee"].unique())}')
print(f'\nDistribution target train :')
print(df_train[TARGET].value_counts().to_string())
print(f'\nDistribution target test :')
print(df_test[TARGET].value_counts().to_string())

X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_test  = df_test[FEATURES]
y_test  = df_test[TARGET]


Train : 248 lignes | annees : [2007, 2012]
Test  : 124  lignes | annees : [2017]

Distribution target train :
bloc_dominant
Gauche    169
Droite     79

Distribution target test :
bloc_dominant
Gauche    69
Droite    32
Centre    23


## 4. Modèle Random Forest

In [6]:
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'Random Forest — Accuracy (test 2017) : {acc_rf:.3f}')
print()
print(classification_report(y_test, y_pred_rf, zero_division=0))


Random Forest — Accuracy (test 2017) : 0.661

              precision    recall  f1-score   support

      Centre       0.00      0.00      0.00        23
      Droite       1.00      0.41      0.58        32
      Gauche       0.62      1.00      0.77        69

    accuracy                           0.66       124
   macro avg       0.54      0.47      0.45       124
weighted avg       0.60      0.66      0.58       124



In [7]:
# Importance des features
feat_imp = pd.Series(
    rf_pipeline.named_steps['model'].feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print('Top 10 features importances (Random Forest):')
print(feat_imp.head(10).round(4).to_string())

Top 10 features importances (Random Forest):
pct_gauche_lag1            0.3034
pct_droite_trend           0.1817
pct_droite_lag1            0.1324
pct_gauche_trend           0.0971
pct_sans_diplome           0.0538
pct_extremedroite_trend    0.0349
employes_pct               0.0345
pct_centre_trend           0.0335
pct_centre_lag1            0.0259
ouvriers_pct               0.0259


## 5. Modèle Gradient Boosting (XGBoost-like via sklearn)

In [8]:
# Encoder les labels pour GBM
all_blocs = ['Centre', 'Divers', 'Droite', 'ExtremeDroite', 'Gauche']
le = LabelEncoder()
le.fit(all_blocs)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

gbm = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
gbm.fit(X_train_imp, y_train_enc)
y_pred_gbm_enc = gbm.predict(X_test_imp)
y_pred_gbm = le.inverse_transform(y_pred_gbm_enc)

acc_gbm = accuracy_score(y_test, y_pred_gbm)
print(f'GradientBoosting — Accuracy (test 2017) : {acc_gbm:.3f}')
print()
print(classification_report(y_test, y_pred_gbm, zero_division=0))


GradientBoosting — Accuracy (test 2017) : 0.661

              precision    recall  f1-score   support

      Centre       0.00      0.00      0.00        23
      Droite       1.00      0.41      0.58        32
      Gauche       0.62      1.00      0.77        69

    accuracy                           0.66       124
   macro avg       0.54      0.47      0.45       124
weighted avg       0.60      0.66      0.58       124



In [9]:
# Choisir le meilleur modèle
best_model_name = 'RandomForest' if acc_rf >= acc_gbm else 'GradientBoosting'
best_pipeline   = rf_pipeline if acc_rf >= acc_gbm else None
best_imputer    = imputer
best_model      = rf_pipeline.named_steps['model'] if acc_rf >= acc_gbm else gbm

print(f'Meilleur modèle : {best_model_name} (accuracy={max(acc_rf, acc_gbm):.3f})')

Meilleur modèle : RandomForest (accuracy=0.661)


## 6. Prédictions 2025, 2026, 2027

**Stratégie** : Pour chaque commune, on extrapole les tendances observées entre 2017 et 2022.
- `pct_X_lag1` = valeur 2022
- `pct_X_trend` = tendance linéaire entre 2017→2022
- Features structurelles (CSP, diplômes) = constantes (2022)
- Démographie = extrapolation linéaire


In [10]:
# Construire les features pour predire 2022
# Lag = resultats 2017 (derniere election connue avant 2022)
# Tendance = evolution 2012 -> 2017
df_2017 = df[df['annee'] == 2017].copy()
df_2012 = df[df['annee'] == 2012].copy().set_index('code_commune')

print(f'Communes avec donnees 2017 : {len(df_2017)}')
print(f'Communes avec donnees 2012 : {len(df_2012)}')

predictions_rows = []

for _, row in df_2017.iterrows():
    comm = row['code_commune']

    def get_trend_2017(col):
        val_2017 = row.get(col, np.nan)
        if comm in df_2012.index:
            val_2012 = df_2012.loc[comm, col]
            if pd.notna(val_2017) and pd.notna(val_2012):
                return float(val_2017) - float(val_2012)
        return 0.0

    feat_row = {
        # Lag = resultats 2017
        'pct_gauche_lag1':         row.get('pct_gauche', np.nan),
        'pct_centre_lag1':         row.get('pct_centre', np.nan),
        'pct_droite_lag1':         row.get('pct_droite', np.nan),
        'pct_extremedroite_lag1':  row.get('pct_extremedroite', np.nan),
        # Tendance 2012 -> 2017
        'pct_gauche_trend':        get_trend_2017('pct_gauche'),
        'pct_centre_trend':        get_trend_2017('pct_centre'),
        'pct_droite_trend':        get_trend_2017('pct_droite'),
        'pct_extremedroite_trend': get_trend_2017('pct_extremedroite'),
        # CSP
        'cadres_pct':              row.get('cadres_pct', np.nan),
        'ouvriers_pct':            row.get('ouvriers_pct', np.nan),
        'employes_pct':            row.get('employes_pct', np.nan),
        'artisans_pct':            row.get('artisans_pct', np.nan),
        # Diplomes
        'pct_bac_plus':            row.get('pct_bac_plus', np.nan),
        'pct_sans_diplome':        row.get('pct_sans_diplome', np.nan),
        # Demographie
        'nb_naissances':           row.get('nb_naissances', np.nan),
        'nb_deces':                row.get('nb_deces', np.nan),
        'solde_naturel':           row.get('solde_naturel', np.nan),
        # Temporel
        'annee':                   2022,
        # Metadata
        '_code_commune':           comm,
        '_libelle':                row.get('libelle', ''),
        '_code_dep':               row.get('code_dep', ''),
    }
    predictions_rows.append(feat_row)

df_pred_input = pd.DataFrame(predictions_rows)
print(f'Communes a predire pour 2022 : {len(df_pred_input)}')
df_pred_input.head(3)


Communes avec donnees 2017 : 124
Communes avec donnees 2012 : 124
Communes a predire pour 2022 : 124


,pct_gauche_lag1,pct_centre_lag1,pct_droite_lag1,pct_extremedroite_lag1,pct_gauche_trend,pct_centre_trend,pct_droite_trend,pct_extremedroite_trend,cadres_pct,ouvriers_pct,...,artisans_pct,pct_bac_plus,pct_sans_diplome,nb_naissances,nb_deces,solde_naturel,annee,_code_commune,_libelle,_code_dep
0,30.634,35.344,28.896,4.990,-20.410,26.007,-4.527,-1.207,NaN,NaN,...,NaN,65.439,11.511,27419.0,63756.0,-36337.0,2022,75056,PARIS,75
1,26.322,34.842,31.026,7.614,-19.497,21.981,-2.147,-0.533,54.351,4.920,...,3.784,63.620,10.122,826.0,373.0,453.0,2022,92002,ANTONY,92
2,28.670,34.321,29.968,6.874,-18.361,23.704,-4.202,-1.308,50.890,6.788,...,4.150,58.683,13.943,1519.0,484.0,1035.0,2022,92004,ASNIERES SUR SEINE,92


In [11]:
# Predire 2022 avec le meilleur modele
X_pred = df_pred_input[FEATURES]

if best_model_name == 'RandomForest':
    y_pred_future      = rf_pipeline.predict(X_pred)
    y_pred_future_prob = rf_pipeline.predict_proba(X_pred)
    classes = rf_pipeline.classes_
else:
    X_pred_imp = best_imputer.transform(X_pred)
    y_pred_future_enc  = gbm.predict(X_pred_imp)
    y_pred_future      = le.inverse_transform(y_pred_future_enc)
    y_pred_future_prob = gbm.predict_proba(X_pred_imp)
    classes = le.classes_

# Construire DataFrame resultats
df_results = df_pred_input[['_code_commune', '_libelle', '_code_dep']].copy()
df_results.columns = ['code_commune', 'libelle', 'code_dep']
df_results['annee'] = 2022
df_results['bloc_predit'] = y_pred_future

for i, cls in enumerate(classes):
    df_results[f'prob_{cls.lower()}'] = y_pred_future_prob[:, i].round(3)

print(f'Predictions 2022 generees : {len(df_results)}')
print('\nDistribution predictions 2022 :')
print(df_results['bloc_predit'].value_counts().to_string())
df_results.head(6)


Predictions 2022 generees : 124

Distribution predictions 2022 :
bloc_predit
Gauche    77
Droite    47


,code_commune,libelle,code_dep,annee,bloc_predit,prob_droite,prob_gauche
0,75056,PARIS,75,2022,Gauche,0.435,0.565
1,92002,ANTONY,92,2022,Droite,0.684,0.316
2,92004,ASNIERES SUR SEINE,92,2022,Gauche,0.473,0.527
3,92007,BAGNEUX,92,2022,Gauche,0.096,0.904
4,92009,BOIS COLOMBES,92,2022,Droite,0.588,0.412
5,92012,BOULOGNE BILLANCOURT,92,2022,Droite,0.757,0.243


## 7. Sauvegarde des prédictions en base

In [12]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS gold.predictions_2022;
        CREATE TABLE gold.predictions_2022 (
            code_commune       CHAR(5)      NOT NULL,
            libelle            VARCHAR(100),
            code_dep           CHAR(3),
            annee              SMALLINT     NOT NULL,
            bloc_predit        VARCHAR(20)  NOT NULL,
            prob_gauche        NUMERIC(6,4),
            prob_centre        NUMERIC(6,4),
            prob_droite        NUMERIC(6,4),
            prob_extremedroite NUMERIC(6,4),
            modele             VARCHAR(30),
            created_at         TIMESTAMP DEFAULT NOW(),
            CONSTRAINT pk_pred_pc PRIMARY KEY (code_commune, annee)
        );
    """))

# Ajouter colonne modele
df_results['modele'] = best_model_name

# Harmoniser les colonnes prob
for col in ['prob_gauche', 'prob_centre', 'prob_droite', 'prob_extremedroite']:
    if col not in df_results.columns:
        df_results[col] = np.nan

insert_cols = [
    'code_commune', 'libelle', 'code_dep', 'annee', 'bloc_predit',
    'prob_gauche', 'prob_centre', 'prob_droite', 'prob_extremedroite', 'modele'
]
df_results[insert_cols].to_sql(
    'predictions_2022',
    engine,
    schema='gold',
    if_exists='append',
    index=False,
    method='multi',
    chunksize=500
)

with engine.connect() as conn:
    n = conn.execute(text('SELECT COUNT(*) FROM gold.predictions_2022')).scalar()
print(f'gold.predictions_2022 : {n} lignes inserees')


gold.predictions_2022 : 124 lignes inserees


## 8. Résultats finaux

In [13]:
print('=' * 65)
print('RESULTATS MODELISATION PETITE COURONNE')
print('=' * 65)
print(f'  Modele Random Forest    — accuracy test 2017 : {acc_rf:.3f}')
print(f'  Modele GradientBoosting — accuracy test 2017 : {acc_gbm:.3f}')
print(f'  Modele retenu : {best_model_name}')
print()
print(f'  Predictions 2022 : {len(df_results)} communes')
dist = df_results['bloc_predit'].value_counts()
for bloc, cnt in dist.items():
    print(f'       {bloc:20s} : {cnt} communes ({cnt/len(df_results)*100:.1f}%)')
print('=' * 65)


RESULTATS MODELISATION PETITE COURONNE
  Modele Random Forest    — accuracy test 2017 : 0.661
  Modele GradientBoosting — accuracy test 2017 : 0.661
  Modele retenu : RandomForest

  Predictions 2022 : 124 communes
       Gauche               : 77 communes (62.1%)
       Droite               : 47 communes (37.9%)


In [14]:
# Repartition predictions 2022 par departement
print('Predictions 2022 par departement :')
print(
    df_results.groupby(['code_dep', 'bloc_predit'])
    .size()
    .unstack(fill_value=0)
    .to_string()
)


Predictions 2022 par departement :
bloc_predit  Droite  Gauche
code_dep                   
75                0       1
92               23      13
93                3      37
94               21      26
